# Token Interactions

Move beyond single-token attribution to understand how **pairs** of input
tokens interact to produce model predictions. Do they provide redundant or
synergistic information? How does one token's importance change when another
is present?

This notebook covers the `irtk.token_interactions` module.

## Why This Matters

Token interactions measure how pairs of input tokens jointly influence predictions — capturing synergies (tokens that matter more together than apart) and redundancies (tokens with overlapping contributions). This reveals the combinatorial structure of model computations.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import token_interactions

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads")

## 1. Token Interaction Matrix

Which tokens matter most for the prediction? Leave-one-out ablation
measures each token's individual contribution.

In [ ]:
prompt = "The Eiffel Tower is in"
tokens = model.to_tokens(prompt)
token_strs = [tokenizer.decode([int(t)]) for t in tokens]

paris_id = tokenizer.encode(" Paris")[0]
def metric(logits):
    return float(logits[-1, paris_id])

# Per-token importance
attr = token_interactions.token_interaction_matrix(model, tokens, metric)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(tokens)), attr, color='steelblue')
ax.set_xticks(range(len(tokens)))
ax.set_xticklabels(token_strs, rotation=45, ha='right')
ax.set_ylabel('|Δ metric| when ablated')
ax.set_title('Token Importance for " Paris" prediction')
plt.tight_layout()
plt.show()

for i, (tok, a) in enumerate(zip(token_strs, attr)):
    print(f"  {tok:>12s}: importance={a:.4f}")

## 2. Pairwise Synergy

Do tokens work together synergistically (joint effect > sum of individual)
or provide redundant information (joint < sum)?

In [ ]:
result = token_interactions.pairwise_synergy(model, tokens, metric)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Synergy matrix
im = axes[0].imshow(result['synergy_matrix'], cmap='RdBu_r', aspect='auto')
axes[0].set_xticks(range(len(tokens)))
axes[0].set_xticklabels(token_strs, rotation=45, ha='right')
axes[0].set_yticks(range(len(tokens)))
axes[0].set_yticklabels(token_strs)
axes[0].set_title('Pairwise Synergy')
plt.colorbar(im, ax=axes[0], label='Synergy (+ synergistic, - redundant)')

# Individual effects
axes[1].bar(range(len(tokens)), result['individual_effects'], color='coral')
axes[1].set_xticks(range(len(tokens)))
axes[1].set_xticklabels(token_strs, rotation=45, ha='right')
axes[1].set_ylabel('Individual effect (signed)')
axes[1].set_title('Individual Token Effects')

plt.suptitle('Token Interaction Analysis', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Conditional Attribution

How does one token's importance change when another is ablated?
If token A becomes more important when token B is removed,
they may carry overlapping information.

In [ ]:
# Check how 'Eiffel' importance changes when 'Tower' is ablated
eiffel_pos = 1  # position of 'Eiffel'
tower_pos = 2   # position of 'Tower'

cond = token_interactions.conditional_attribution(
    model, tokens, metric, target_pos=eiffel_pos, context_pos=tower_pos
)

print(f"Token: '{token_strs[eiffel_pos]}', Context: '{token_strs[tower_pos]}'")
print(f"  Effect of ablating '{token_strs[eiffel_pos]}' alone:  {cond['effect_target']:.4f}")
print(f"  Effect of ablating '{token_strs[tower_pos]}' alone:  {cond['effect_context']:.4f}")
print(f"  Effect of ablating both:              {cond['effect_both']:.4f}")
print(f"  Conditional effect (target|no context): {cond['conditional_effect']:.4f}")
print(f"  Importance change:                      {cond['importance_change']:.4f}")

# Scan all pairs
print("\nConditional importance changes:")
for i in range(len(tokens)):
    for j in range(len(tokens)):
        if i == j:
            continue
        c = token_interactions.conditional_attribution(
            model, tokens, metric, target_pos=i, context_pos=j
        )
        if abs(c['importance_change']) > 0.1:
            sign = '+' if c['importance_change'] > 0 else '-'
            print(f"  {token_strs[i]:>10s} | no {token_strs[j]:>10s}: "
                  f"change={c['importance_change']:+.4f} ({sign}dependent)")

## 4. Bigram Attention Scores

Which token pairs are most strongly linked by attention across all heads?

In [ ]:
bigram = token_interactions.bigram_attention_scores(model, tokens)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(bigram, cmap='Blues', aspect='auto')
ax.set_xticks(range(len(tokens)))
ax.set_xticklabels(token_strs, rotation=45, ha='right')
ax.set_yticks(range(len(tokens)))
ax.set_yticklabels(token_strs)
ax.set_xlabel('Key (attended to)')
ax.set_ylabel('Query (attending from)')
ax.set_title('Mean Attention Bigram Scores (all layers)')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 5. Token Pair Effect

Get a detailed breakdown of how two specific tokens jointly affect
the model's prediction, including synergy and redundancy ratio.

In [ ]:
pair = token_interactions.token_pair_effect(
    model, tokens, metric, pos_a=eiffel_pos, pos_b=tower_pos
)

print(f"Tokens: '{token_strs[eiffel_pos]}' + '{token_strs[tower_pos]}'")
print(f"  Clean metric:     {pair['clean_metric']:.4f}")
print(f"  Effect of A only: {pair['effect_a']:.4f}")
print(f"  Effect of B only: {pair['effect_b']:.4f}")
print(f"  Effect of both:   {pair['effect_both']:.4f}")
print(f"  Synergy:          {pair['synergy']:.4f}")
print(f"  Redundancy ratio: {pair['redundancy_ratio']:.4f}")
print(f"    (1.0 = perfectly additive, >1 = synergistic, <1 = redundant)")

## Summary

| Function | Purpose |
|----------|--------|
| `token_interaction_matrix()` | Per-token importance via leave-one-out ablation |
| `pairwise_synergy()` | Synergy/redundancy between all pairs |
| `conditional_attribution()` | How one token's importance changes given another |
| `bigram_attention_scores()` | Attention-weighted co-occurrence across heads |
| `token_pair_effect()` | Detailed two-token joint effect analysis |